# Lab 2 · Exercise C — Pulls, bias and coverage
Test your fit with **toy experiments**: generate with known truth, fit, compare.

In [ ]:
import ROOT
%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
import matplotlib.pyplot as plt

m     = ROOT.RooRealVar("m", "m [MeV]", 5080, 5540)
mean  = ROOT.RooRealVar("mean", "mean", 5288, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 14.8, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = ROOT.RooGaussian("sig", "sig", m, mean, sigma)
bkg   = ROOT.RooExponential("bkg", "bkg", m, slope)
nsig  = ROOT.RooRealVar("nsig", "signal yield", 33, 0, 400)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 133, 0, 600)
model = ROOT.RooAddPdf("model", "sig+bkg", ROOT.RooArgList(sig, bkg), ROOT.RooArgList(nsig, nbkg))

TRUTH = {mean: 5288, sigma: 14.8, slope: -0.002, nbkg: 133}

def reset(ns_true):
    """Put every parameter back to the truth (also used as the fit start)."""
    for v, t in TRUTH.items():
        v.setVal(t)
    nsig.setVal(ns_true)

## C1 · The toy loop
Complete `run_toys`: for each toy, **reset** the parameters to the truth, **generate** an extended dataset from the model,
**fit** it, and record the pull of `nsig` if the fit converged (`status() == 0`).

In [ ]:
def run_toys(ns_true, ntoys=300):
    pulls, values, nfail = [], [], 0
    for i in range(ntoys):
        reset(ns_true)
        data = None   # TODO: model.generate(ROOT.RooArgSet(m), ROOT.RooFit.Extended(True))
        res  = None   # TODO: fit model to data (Extended=True, Save=True, PrintLevel=-1)
        if res.status() != 0:
            nfail += 1
            continue
        values.append(nsig.getVal())
        pulls.append((nsig.getVal() - ns_true) / nsig.getError())
    return np.array(pulls), np.array(values), nfail

In [ ]:
pulls33, vals33, nfail33 = run_toys(33)
print("N toys fitted: %d, failed: %d" % (len(pulls33), nfail33))
print("pull mean = %.2f   RMS = %.2f   coverage |pull|<1: %.2f" %
      (pulls33.mean(), pulls33.std(), np.mean(np.abs(pulls33) < 1)))

plt.hist(pulls33, bins=30, range=(-4, 4), histtype="step")
plt.xlabel("pull of nsig"); plt.ylabel("toys"); plt.title("N_s = 33"); plt.show()

In [ ]:
check("enough toys converged", len(pulls33) > 250)
check("no large bias: |mean pull| < 0.3", abs(pulls33.mean()) < 0.3)
check("errors have the right size: 0.8 < RMS < 1.2", 0.8 < pulls33.std() < 1.2)

## C2 · Low statistics
Repeat with $N_s=5$. Look at the pull distribution, the fraction of fits with `nsig` at the lower boundary (nsig < 0.01) and the coverage.

In [ ]:
pulls5, vals5, nfail5 = run_toys(5)
# TODO: print mean, RMS, coverage |pull|<1, failed fits, and the fraction of toys with nsig < 0.01
# TODO: histogram of the pulls

## C3 · (optional) The same toy loop in C++
The loop is declared in C++ with `gInterpreter.Declare` and called from Python. Run it and compare the time and the pulls with the Python loop.

In [ ]:
import time
ROOT.gInterpreter.Declare("""
#include "RooAbsPdf.h"
#include "RooArgSet.h"
#include "RooDataSet.h"
#include "RooFitResult.h"
#include "RooGlobalFunc.h"
#include "RooRealVar.h"
#include <memory>
#include <vector>

std::vector<double> toy_pulls(RooAbsPdf &model, RooRealVar &m, RooRealVar &nsig,
                              RooArgSet &params, double ns_true, int ntoys) {
  std::vector<double> pulls;
  std::unique_ptr<RooArgSet> start(static_cast<RooArgSet *>(params.snapshot()));
  for (int i = 0; i < ntoys; ++i) {
    params.assign(*start);
    nsig.setVal(ns_true);
    std::unique_ptr<RooDataSet> data(model.generate(RooArgSet(m), RooFit::Extended(true)));
    std::unique_ptr<RooFitResult> res(model.fitTo(*data, RooFit::Extended(true), RooFit::Save(true),
                                                  RooFit::PrintLevel(-1)));
    if (res->status() == 0) pulls.push_back((nsig.getVal() - ns_true) / nsig.getError());
  }
  return pulls;
}
""")

reset(33)
params = ROOT.RooArgSet(mean, sigma, slope, nsig, nbkg)
t0 = time.time(); pulls_py = run_toys(33, 100)[0]; t_py = time.time() - t0
reset(33)
t0 = time.time(); pulls_cpp = np.array(list(ROOT.toy_pulls(model, m, nsig, params, 33.0, 100))); t_cpp = time.time() - t0
print("Python loop: %.1f s   C++ loop: %.1f s   (100 toys each)" % (t_py, t_cpp))
print("pull RMS: Python %.2f  C++ %.2f" % (pulls_py.std(), pulls_cpp.std()))

In [ ]:
check("the C++ loop gives sensible pulls", 0.6 < pulls_cpp.std() < 1.4 and abs(pulls_cpp.mean()) < 0.5)

## Questions (2–3 lines each)
1. For $N_s=33$: is the fit biased? Are the HESSE errors reliable?
2. For $N_s=5$: what is different in the pull distribution, and why? What would you report instead of a symmetric error?
3. Why are failed fits important to count? What bias do you introduce by discarding them?